In [ ]:
running_date = "2026-07-20"
seed = True  # POC: seed synthetic bronze before running. Set False in prod.


In [ ]:
%run pipeline_service.py


In [ ]:
%run simulators_service.py


In [ ]:
def get_secret(name):
    return spark.conf.get("spark.ssv.secret." + name.replace("-", "_"))


In [ ]:
for s in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")


In [ ]:
if seed:
    save_service_bronze(spark, running_date)


In [ ]:
# Bronze is fed EITHER by the synthetic seed above (seed=True) OR by the pipeline's
# Copy activities (seed=False) — never by the notebook's own ingest here, so always
# with_ingest=False. (service_bronze_ingest stays available for a pure-notebook path.)
pipe = SaleServicePipeline(secret=get_secret, schema_enabled=True)
pipe.run(running_date, with_ingest=False)
run_service_dq(pipe.context(running_date))
